# 📐 Banking Credit Risk & Fraud Detection Analytics

## Notebook 04 — Statistical Analysis

### Objective

The objective of this notebook is to statistically validate selected relationships identified during Exploratory Data Analysis (EDA).

EDA helps us identify patterns, but statistical hypothesis testing helps determine whether the observed differences or relationships provide sufficient evidence against a specified null hypothesis.

The analysis will cover:

- Hypothesis testing fundamentals
- Null and alternative hypotheses
- Significance level
- p-value
- Confidence intervals
- Effect size
- Categorical-variable testing
- Numerical-variable testing
- Credit-risk statistical analysis
- Fraud-detection statistical analysis

### Analytical Framework

For each statistical test, the following process will be followed:

1. Define the business question.
2. Define the null hypothesis (H₀).
3. Define the alternative hypothesis (H₁).
4. Select an appropriate statistical test.
5. Set the significance level (α).
6. Calculate the test statistic and p-value.
7. Evaluate the result.
8. Calculate an effect size where appropriate.
9. Translate the result into a business interpretation.

### Important Principle

Statistical significance does not automatically mean practical or business significance.

A statistically significant relationship should therefore be evaluated together with its effect size, direction, and business context.

## 1. Import Statistical Analysis Libraries

The following libraries will be used for statistical testing and numerical analysis.

In [1]:
import pandas as pd
import numpy as np

from scipy import stats

print("Statistical libraries imported successfully!")

Statistical libraries imported successfully!


## 2. Load Processed Datasets

The statistical analysis will use the cleaned datasets generated during the data-cleaning stage.

Using processed data ensures that the analysis is performed consistently with the data used during EDA.

In [2]:
credit_df = pd.read_csv(
    "../data/processed/credit_risk_cleaned.csv"
)

fraud_df = pd.read_csv(
    "../data/processed/fraud_cleaned.csv"
)

print("Credit Risk Shape:", credit_df.shape)
print("Fraud Detection Shape:", fraud_df.shape)

Credit Risk Shape: (32416, 13)
Fraud Detection Shape: (283726, 31)


## 3. Hypothesis Testing Fundamentals

Hypothesis testing is a statistical framework used to evaluate whether observed evidence is sufficiently strong to reject a predefined null hypothesis.

### Null Hypothesis — H₀

The null hypothesis represents the default assumption.

It generally states that:

- There is no difference.
- There is no association.
- There is no effect.

### Alternative Hypothesis — H₁

The alternative hypothesis represents the possibility that a meaningful difference, association, or effect exists.

### Significance Level — α

The significance level represents the threshold used to determine whether the evidence against H₀ is sufficiently strong.

For this project:

**α = 0.05**

### P-value

The p-value represents the probability of observing a result at least as extreme as the observed result, assuming that the null hypothesis is true.

Decision rule:

**p-value < α → Reject H₀**

**p-value ≥ α → Fail to reject H₀**

### Important Terminology

We should say:

> "Fail to reject H₀"

rather than:

> "Accept H₀"

because statistical testing does not prove that the null hypothesis is true.

## 4. Hypothesis Test — Loan-to-Income Ratio

### Business Question

Is the average `loan_percent_income` different between defaulted and non-defaulted applicants?

### Hypotheses

**H₀:** The mean loan-to-income ratio is the same for defaulted and non-defaulted applicants.

**H₁:** The mean loan-to-income ratio is different between defaulted and non-defaulted applicants.

This is a two-sided hypothesis because we are testing for a difference in either direction.

### Significance Level

α = 0.05

### Statistical Test

An independent two-sample t-test will initially be used because we are comparing the means of a numerical variable across two independent groups.

Because the two groups may have different variances, Welch's t-test will be used rather than assuming equal variances.

In [3]:
no_default = credit_df.loc[
    credit_df["loan_status"] == 0,
    "loan_percent_income"
]

default = credit_df.loc[
    credit_df["loan_status"] == 1,
    "loan_percent_income"
]

print("No Default observations:", len(no_default))
print("Default observations:", len(default))

No Default observations: 25327
Default observations: 7089


In [4]:
t_stat, p_value = stats.ttest_ind(
    no_default,
    default,
    equal_var=False,
    nan_policy="omit"
)

print("T-statistic:", t_stat)
print("P-value:", p_value)

T-statistic: -59.031952983037094
P-value: 0.0


## 4.1 Hypothesis Test Decision

In [5]:
alpha = 0.05

if p_value < alpha:
    print("Reject H0")
    print("There is statistically significant evidence of a difference.")
else:
    print("Fail to reject H0")
    print("There is insufficient statistical evidence of a difference.")

Reject H0
There is statistically significant evidence of a difference.


## 4.2 Effect Size — Cohen's d

Statistical significance can be strongly influenced by sample size.

With a large dataset, even a relatively small difference can produce a very small p-value.

Therefore, effect size is used to evaluate the practical magnitude of the difference.

Cohen's d measures the standardized difference between two group means.

A commonly used interpretation is:

- ~0.2 → small effect
- ~0.5 → medium effect
- ~0.8 or higher → large effect

These thresholds are guidelines rather than strict rules.

In [6]:
n1 = no_default.count()
n2 = default.count()

mean1 = no_default.mean()
mean2 = default.mean()

std1 = no_default.std()
std2 = default.std()

pooled_std = np.sqrt(
    (
        ((n1 - 1) * std1**2) +
        ((n2 - 1) * std2**2)
    )
    / (n1 + n2 - 2)
)

cohens_d = (mean2 - mean1) / pooled_std

print("Cohen's d:", cohens_d)

Cohen's d: 0.9928982271035547


### 🔎 Effect Size Finding

Cohen's d for the difference in `loan_percent_income` between defaulted and non-defaulted applicants is approximately **0.993**.

This represents a large standardized effect based on commonly used Cohen's d guidelines.

The result indicates that the difference in loan-to-income ratio between the two groups is not only statistically testable but also substantial in standardized terms.

The earlier EDA showed that the mean loan-to-income ratio was approximately:

- Non-default: 0.149
- Default: 0.247

Therefore, the higher loan-to-income ratio observed among defaulted applicants represents a potentially meaningful credit-risk signal.

However, effect size alone does not establish statistical significance. The t-test p-value must also be considered before making the final hypothesis-test decision.

In [7]:
print("T-statistic:", t_stat)
print("P-value:", p_value)
print("Cohen's d:", cohens_d)

T-statistic: -59.031952983037094
P-value: 0.0
Cohen's d: 0.9928982271035547


## 4.3 Statistical Conclusion

### Hypotheses

**H₀:** The mean loan-to-income ratio is the same for defaulted and non-defaulted applicants.

**H₁:** The mean loan-to-income ratio differs between defaulted and non-defaulted applicants.

### Test Results

- Test: Welch's independent two-sample t-test
- T-statistic: **-59.032**
- P-value: **< 0.001** (reported by Python as 0.0 due to numerical precision)
- Significance level: **α = 0.05**
- Cohen's d: **0.993**

### Decision

Since the p-value is substantially below the significance level of 0.05, the null hypothesis is rejected.

There is statistically significant evidence that the mean loan-to-income ratio differs between defaulted and non-defaulted applicants.

The Cohen's d value of approximately 0.993 indicates a large standardized effect.

### Business Interpretation

Defaulted applicants have a substantially higher loan-to-income ratio than non-defaulted applicants.

The EDA showed:

- Non-default mean: **0.149**
- Default mean: **0.247**

The statistical test provides strong evidence that this observed difference is unlikely to be explained by random sampling variation alone.

The large effect size further indicates that the difference is practically meaningful within this dataset.

However, statistical significance does not establish causation. Other factors such as income, loan amount, interest rate, loan grade, and employment characteristics may also contribute to default risk.

In [8]:
stats.ttest_ind(
    no_default,
    default,
    equal_var=False
)

TtestResult(statistic=np.float64(-59.031952983037094), pvalue=np.float64(0.0), df=np.float64(8887.513510265479))

In [9]:
equal_var=False

# 5. Chi-Square Test — Home Ownership and Default

### Business Question

Is applicant home-ownership status associated with loan default?

### Variables

**Categorical Variable 1:** `person_home_ownership`

**Categorical Variable 2:** `loan_status`

### Hypotheses

**H₀:** Home ownership and loan default are independent.

**H₁:** Home ownership and loan default are associated.

### Statistical Test

A Chi-Square Test of Independence will be used because both variables are categorical.

### Significance Level

α = 0.05

### Important Principle

The Chi-Square test evaluates whether the observed distribution of one categorical variable differs across the categories of another categorical variable.

It does not establish causation.

In [10]:
home_ownership_table = pd.crosstab(
    credit_df["person_home_ownership"],
    credit_df["loan_status"]
)

home_ownership_table

loan_status,0,1
person_home_ownership,,
MORTGAGE,11682,1687
OTHER,73,33
OWN,2371,192
RENT,11201,5177


In [11]:
chi2, p_value, degrees_of_freedom, expected = stats.chi2_contingency(
    home_ownership_table
)

print("Chi-square statistic:", chi2)
print("P-value:", p_value)
print("Degrees of freedom:", degrees_of_freedom)

Chi-square statistic: 1894.3526733955891
P-value: 0.0
Degrees of freedom: 3


## 5.1 Expected Frequencies

The Chi-Square test compares the observed frequencies with the frequencies that would be expected if the two categorical variables were independent.

The expected frequency table is examined to verify that the assumptions of the Chi-Square test are reasonably satisfied.

In [12]:
expected_table = pd.DataFrame(
    expected,
    index=home_ownership_table.index,
    columns=home_ownership_table.columns
)

expected_table

loan_status,0,1
person_home_ownership,,
MORTGAGE,10445.356090,2923.643910
OTHER,82.819040,23.180960
OWN,2002.501882,560.498118
RENT,12796.322989,3581.677011


In [13]:
print("Chi-square statistic:", chi2)
print("P-value:", p_value)
print("Degrees of freedom:", degrees_of_freedom)

Chi-square statistic: 1894.3526733955891
P-value: 0.0
Degrees of freedom: 3


## 5.2 Chi-Square Test Result

### Business Question

Is applicant home ownership associated with loan default?

### Hypotheses

**H₀:** Home ownership and loan default are independent.

**H₁:** Home ownership and loan default are associated.

### Test Results

- Test: Chi-Square Test of Independence
- Chi-square statistic: **1894.353**
- Degrees of freedom: **3**
- P-value: **< 0.001**
- Significance level: **α = 0.05**

### Decision

Since the p-value is substantially below 0.05, we reject the null hypothesis.

There is statistically significant evidence of an association between home ownership and loan default in this dataset.

### Business Interpretation

The observed default rates differ substantially across home-ownership categories.

Applicants classified as `RENT` have an observed default rate of approximately 31.61%, while applicants classified as `OWN` have an observed default rate of approximately 7.49%.

The statistical test confirms that the distribution of loan default differs across home-ownership categories.

However, statistical association does not establish causation. Other characteristics such as income, loan grade, loan amount, and employment history may contribute to the observed relationship.

## 5.3 Effect Size — Cramér's V

A statistically significant Chi-Square test tells us that an association exists, but it does not tell us how strong that association is.

Cramér's V is used to measure the strength of association between two categorical variables.

Its value ranges from:

- 0 → no association
- 1 → perfect association

The effect-size interpretation should be treated as contextual rather than as a rigid universal threshold.

In [14]:
n = home_ownership_table.to_numpy().sum()

min_dimension = min(
    home_ownership_table.shape
) - 1

cramers_v = np.sqrt(
    chi2 / (
        n * min_dimension
    )
)

print("Cramér's V:", cramers_v)

Cramér's V: 0.2417412178963734


# 6. Chi-Square Test — Loan Grade and Default

### Business Question

Is loan grade associated with loan default?

### Variables

**Categorical Variable 1:** `loan_grade`

**Categorical Variable 2:** `loan_status`

### Hypotheses

**H₀:** Loan grade and loan default are independent.

**H₁:** Loan grade and loan default are associated.

### Statistical Test

A Chi-Square Test of Independence will be used because both variables are categorical.

### Significance Level

α = 0.05

The EDA showed a strong difference in observed default rates across loan grades. This statistical test will determine whether the observed association is statistically significant.

In [15]:
loan_grade_table = pd.crosstab(
    credit_df["loan_grade"],
    credit_df["loan_status"]
)

loan_grade_table

loan_status,0,1
loan_grade,,
A,9637,1066
B,8692,1695
C,5102,1336
D,1482,2138
E,342,621
F,71,170
G,1,63


In [16]:
chi2_grade, p_grade, dof_grade, expected_grade = (
    stats.chi2_contingency(
        loan_grade_table
    )
)

print("Chi-square statistic:", chi2_grade)
print("P-value:", p_grade)
print("Degrees of freedom:", dof_grade)

Chi-square statistic: 5588.326418998944
P-value: 0.0
Degrees of freedom: 6


## 6.1 Expected Frequencies

The expected frequencies are examined to verify that the Chi-Square test assumptions are reasonably satisfied.

Expected frequencies represent the counts we would expect if loan grade and loan default were independent.

In [18]:
expected_grade_table = pd.DataFrame(
    expected_grade,
    index=loan_grade_table.index,
    columns=loan_grade_table.columns
)

expected_grade_table

loan_status,0,1
loan_grade,,
A,8362.379103,2340.620897
B,8115.484606,2271.515394
C,5030.084711,1407.915289
D,2828.348346,791.651654
E,752.403165,210.596835
F,188.296119,52.703881
G,50.003949,13.996051


## 6.2 Effect Size — Cramér's V

Because the Chi-Square statistic is influenced by sample size, Cramér's V will be calculated to measure the strength of the association between loan grade and loan default.

A larger Cramér's V indicates a stronger association.

In [19]:
n_grade = loan_grade_table.to_numpy().sum()

min_dimension_grade = (
    min(loan_grade_table.shape) - 1
)

cramers_v_grade = np.sqrt(
    chi2_grade /
    (n_grade * min_dimension_grade)
)

print("Cramér's V:", cramers_v_grade)

Cramér's V: 0.41520365796210246


# 7. Hypothesis Test — Interest Rate and Default

### Business Question

Is the mean interest rate different between defaulted and non-defaulted loans?

### Hypotheses

**H₀:** The mean interest rate is the same for defaulted and non-defaulted loans.

**H₁:** The mean interest rate differs between defaulted and non-defaulted loans.

### Statistical Test

Welch's independent two-sample t-test will be used because:

- Interest rate is numerical.
- Loan status contains two independent groups.
- Equal population variances are not assumed.

### Significance Level

α = 0.05

In [20]:
interest_no_default = credit_df.loc[
    credit_df["loan_status"] == 0,
    "loan_int_rate"
].dropna()

interest_default = credit_df.loc[
    credit_df["loan_status"] == 1,
    "loan_int_rate"
].dropna()

print(
    "No Default observations:",
    len(interest_no_default)
)

print(
    "Default observations:",
    len(interest_default)
)

No Default observations: 25327
Default observations: 7089


In [21]:
t_interest, p_interest = stats.ttest_ind(
    interest_no_default,
    interest_default,
    equal_var=False
)

print("T-statistic:", t_interest)
print("P-value:", p_interest)

T-statistic: -56.93830491507941
P-value: 0.0


In [22]:
n1 = interest_no_default.count()
n2 = interest_default.count()

mean1 = interest_no_default.mean()
mean2 = interest_default.mean()

std1 = interest_no_default.std()
std2 = interest_default.std()

pooled_std = np.sqrt(
    (
        ((n1 - 1) * std1**2) +
        ((n2 - 1) * std2**2)
    )
    / (n1 + n2 - 2)
)

cohens_d_interest = (
    (mean2 - mean1) / pooled_std
)

print(
    "Cohen's d:",
    cohens_d_interest
)

Cohen's d: 0.8173218131407798


In [23]:
alpha = 0.05

if p_interest < alpha:
    print("Reject H0")
    print(
        "There is statistically significant "
        "evidence of a difference in interest rates."
    )
else:
    print("Fail to reject H0")
    print(
        "There is insufficient statistical "
        "evidence of a difference."
    )

Reject H0
There is statistically significant evidence of a difference in interest rates.


# 8. Statistical Significance of Correlation

### Business Question

Is there statistically significant evidence of a linear relationship between loan interest rate and loan default?

### Hypotheses

**H₀:** There is no linear correlation between `loan_int_rate` and `loan_status` in the population.

**H₁:** There is a linear correlation between `loan_int_rate` and `loan_status` in the population.

### Statistical Test

Pearson correlation significance testing will be used.

The test produces:

- Pearson correlation coefficient (`r`)
- p-value

### Significance Level

α = 0.05

### Important Limitation

Because `loan_status` is binary, Pearson correlation should be interpreted carefully. It is mathematically related to the point-biserial correlation in this two-group setting.

The result indicates linear association with the binary outcome; it does not establish causation.

In [24]:
correlation_data = credit_df[
    ["loan_int_rate", "loan_status"]
].dropna()

r_value, p_corr = stats.pearsonr(
    correlation_data["loan_int_rate"],
    correlation_data["loan_status"]
)

print("Pearson correlation:", r_value)
print("P-value:", p_corr)

Pearson correlation: 0.3200812719295369
P-value: 0.0


In [25]:
eda_correlation = credit_df[
    ["loan_int_rate", "loan_status"]
].corr().iloc[0, 1]

print("EDA correlation:", eda_correlation)
print("Statistical test correlation:", r_value)

EDA correlation: 0.3200812719295363
Statistical test correlation: 0.3200812719295369


In [26]:
alpha = 0.05

if p_corr < alpha:
    print("Reject H0")
    print(
        "There is statistically significant evidence "
        "of a linear association."
    )
else:
    print("Fail to reject H0")
    print(
        "There is insufficient evidence "
        "of a linear association."
    )

Reject H0
There is statistically significant evidence of a linear association.


## 8.1 Interpreting Pearson's r

The Pearson correlation coefficient describes both direction and strength of linear association.

- Positive `r` → higher values of one variable tend to occur with higher values of the other.
- Negative `r` → higher values of one variable tend to occur with lower values of the other.
- `r` close to 0 → weak linear association.
- Larger absolute values of `r` → stronger linear association.

The practical interpretation depends on the domain, data quality, sample size, and modeling context.

A statistically significant correlation does not automatically imply that the relationship is strong or practically important.

In [27]:
r_value, p_corr = stats.pearsonr(
    correlation_data["loan_int_rate"],
    correlation_data["loan_status"]
)

print("Pearson correlation:", r_value)
print("P-value:", p_corr)

Pearson correlation: 0.3200812719295369
P-value: 0.0


## 8.2 Statistical Conclusion — Interest Rate and Default

### Test Results

- Pearson correlation: **0.3201**
- P-value: **< 0.001**
- Significance level: **α = 0.05**

### Decision

Since the p-value is substantially below 0.05, we reject the null hypothesis.

There is statistically significant evidence of a positive linear association between `loan_int_rate` and `loan_status` in this dataset.

The correlation coefficient of approximately 0.32 indicates a positive relationship of moderate magnitude.

This means that higher interest rates tend to be associated with a higher probability of observing the default class.

### Business Interpretation

Interest rate appears to be a meaningful risk-related variable in this dataset.

The EDA also showed that defaulted loans had higher average and median interest rates than non-defaulted loans.

However, correlation does not imply causation. Interest rates may themselves reflect other risk characteristics considered during lending decisions.

Therefore, interest rate should be evaluated together with loan grade, income, loan amount, loan-to-income ratio, and other applicant characteristics during model development.

## 9. Correlation Significance — Loan Amount and Default

### Business Question

Is there statistically significant evidence of a linear association between `loan_amnt` and `loan_status`?

### Hypotheses

**H₀:** There is no linear correlation between `loan_amnt` and `loan_status`.

**H₁:** There is a linear correlation between `loan_amnt` and `loan_status`.

### Statistical Test

Pearson correlation significance test.

### Significance Level

α = 0.05

In [28]:
loan_amount_corr_data = credit_df[
    ["loan_amnt", "loan_status"]
].dropna()

r_loan, p_loan = stats.pearsonr(
    loan_amount_corr_data["loan_amnt"],
    loan_amount_corr_data["loan_status"]
)

print("Pearson correlation:", r_loan)
print("P-value:", p_loan)

Pearson correlation: 0.10573628636948634
P-value: 3.0560071961504645e-81


In [29]:
if p_loan < 0.05:
    print("Reject H0")
    print("The correlation is statistically significant.")
else:
    print("Fail to reject H0")
    print("The correlation is not statistically significant.")

Reject H0
The correlation is statistically significant.


## 9.1 Statistical Conclusion — Loan Amount and Default

### Test Results

- Pearson correlation: **0.1057**
- P-value: **3.056 × 10⁻⁸¹**
- Significance level: **α = 0.05**

### Decision

Since the p-value is substantially below 0.05, we reject the null hypothesis.

There is statistically significant evidence of a linear association between `loan_amnt` and `loan_status`.

However, the Pearson correlation coefficient is approximately 0.106, indicating that the linear association is weak.

### Business Interpretation

Larger loan amounts are associated with a higher observed probability of default, but the relationship is relatively weak when considered on its own.

This demonstrates an important distinction between statistical significance and practical significance.

The extremely small p-value is partly influenced by the large sample size, whereas the correlation coefficient indicates that loan amount alone explains only a limited amount of the linear variation in the binary default outcome.

Therefore, `loan_amnt` should not be evaluated independently. Its predictive value should be assessed together with variables such as income, loan-to-income ratio, interest rate, loan grade, and employment characteristics.

In [30]:
print("Chi-square statistic:", chi2_grade)
print("P-value:", p_grade)
print("Degrees of freedom:", dof_grade)
print("Cramér's V:", cramers_v_grade)

Chi-square statistic: 5588.326418998944
P-value: 0.0
Degrees of freedom: 6
Cramér's V: 0.41520365796210246


# 10. Statistical Analysis Summary

The statistical analysis validates selected relationships identified during exploratory data analysis.

The tests demonstrate that statistical significance and practical significance must be evaluated separately.

The following results have been established so far:

1. `loan_percent_income` vs `loan_status`
   - Welch's t-test
   - Statistically significant
   - Large effect size

2. `person_home_ownership` vs `loan_status`
   - Chi-Square Test of Independence
   - Statistically significant
   - Cramér's V used to evaluate association strength

3. `loan_int_rate` vs `loan_status`
   - Welch's t-test
   - Statistically significant difference evaluated using p-value and Cohen's d

4. `loan_int_rate` vs `loan_status`
   - Pearson correlation
   - Statistically significant positive association
   - Correlation magnitude is moderate

5. `loan_amnt` vs `loan_status`
   - Pearson correlation
   - Statistically significant
   - Weak positive linear association

6. `loan_grade` vs `loan_status`
   - Chi-Square Test of Independence
   - Cramér's V used to evaluate association strength

In [31]:
statistical_summary = pd.DataFrame({
    "Analysis": [
        "Loan-to-Income Ratio vs Default",
        "Home Ownership vs Default",
        "Interest Rate vs Default",
        "Interest Rate Correlation",
        "Loan Amount Correlation",
        "Loan Grade vs Default"
    ],
    
    "Test": [
        "Welch's t-test",
        "Chi-Square",
        "Welch's t-test",
        "Pearson correlation",
        "Pearson correlation",
        "Chi-Square"
    ],
    
    "Statistic": [
        t_stat,
        chi2,
        t_interest,
        r_value,
        r_loan,
        chi2_grade
    ],
    
    "P_Value": [
        p_value,
        p_value,
        p_interest,
        p_corr,
        p_loan,
        p_grade
    ],
    
    "Effect_Size": [
        cohens_d,
        cramers_v,
        cohens_d_interest,
        r_value,
        r_loan,
        cramers_v_grade
    ]
})

statistical_summary

,Analysis,Test,Statistic,P_Value,Effect_Size
0,Loan-to-Income Ratio vs Default,Welch's t-test,-59.031953,0.000000e+00,0.992898
1,Home Ownership vs Default,Chi-Square,1894.352673,0.000000e+00,0.241741
2,Interest Rate vs Default,Welch's t-test,-56.938305,0.000000e+00,0.817322
3,Interest Rate Correlation,Pearson correlation,0.320081,0.000000e+00,0.320081
4,Loan Amount Correlation,Pearson correlation,0.105736,3.056007e-81,0.105736
5,Loan Grade vs Default,Chi-Square,5588.326419,0.000000e+00,0.415204


In [32]:
statistical_summary["Decision"] = np.where(
    statistical_summary["P_Value"] < 0.05,
    "Reject H0",
    "Fail to Reject H0"
)

statistical_summary

,Analysis,Test,Statistic,P_Value,Effect_Size,Decision
0,Loan-to-Income Ratio vs Default,Welch's t-test,-59.031953,0.000000e+00,0.992898,Reject H0
1,Home Ownership vs Default,Chi-Square,1894.352673,0.000000e+00,0.241741,Reject H0
2,Interest Rate vs Default,Welch's t-test,-56.938305,0.000000e+00,0.817322,Reject H0
3,Interest Rate Correlation,Pearson correlation,0.320081,0.000000e+00,0.320081,Reject H0
4,Loan Amount Correlation,Pearson correlation,0.105736,3.056007e-81,0.105736,Reject H0
5,Loan Grade vs Default,Chi-Square,5588.326419,0.000000e+00,0.415204,Reject H0


## 11. Multiple Hypothesis Testing

When multiple statistical hypotheses are tested simultaneously, the probability of obtaining at least one statistically significant result by chance increases.

This creates a multiple-comparisons problem.

For exploratory analysis, the standard α = 0.05 threshold may be used as an initial screening threshold.

For a more rigorous confirmatory analysis, multiple-testing correction methods such as Bonferroni or False Discovery Rate (FDR) can be considered.

The correction method should be selected based on the purpose of the analysis and the trade-off between controlling false positives and statistical power.

In [33]:
from statsmodels.stats.multitest import multipletests

p_values = statistical_summary["P_Value"].values

bonferroni_results = multipletests(
    p_values,
    alpha=0.05,
    method="bonferroni"
)

statistical_summary["Bonferroni_P_Value"] = (
    bonferroni_results[1]
)

statistical_summary["Bonferroni_Decision"] = np.where(
    bonferroni_results[0],
    "Reject H0",
    "Fail to Reject H0"
)

statistical_summary

,Analysis,Test,Statistic,P_Value,Effect_Size,Decision,Bonferroni_P_Value,Bonferroni_Decision
0,Loan-to-Income Ratio vs Default,Welch's t-test,-59.031953,0.000000e+00,0.992898,Reject H0,0.000000e+00,Reject H0
1,Home Ownership vs Default,Chi-Square,1894.352673,0.000000e+00,0.241741,Reject H0,0.000000e+00,Reject H0
2,Interest Rate vs Default,Welch's t-test,-56.938305,0.000000e+00,0.817322,Reject H0,0.000000e+00,Reject H0
3,Interest Rate Correlation,Pearson correlation,0.320081,0.000000e+00,0.320081,Reject H0,0.000000e+00,Reject H0
4,Loan Amount Correlation,Pearson correlation,0.105736,3.056007e-81,0.105736,Reject H0,1.833604e-80,Reject H0
5,Loan Grade vs Default,Chi-Square,5588.326419,0.000000e+00,0.415204,Reject H0,0.000000e+00,Reject H0


# 12. Statistical Analysis Conclusion

The statistical analysis formally evaluated several relationships identified during EDA.

The results demonstrate that multiple customer and loan characteristics have statistically significant associations with credit default in this dataset.

However, the analyses also demonstrate an important Data Science principle:

### Statistical Significance ≠ Practical Significance

A very small p-value indicates strong evidence against the null hypothesis, but it does not automatically indicate a strong or practically important relationship.

Effect-size measures such as:

- Cohen's d
- Pearson's r
- Cramér's V

provide additional context regarding the magnitude of observed relationships.

### Key Statistical Insights

- Loan-to-income ratio showed a substantial difference between defaulted and non-defaulted applicants.
- Interest rate showed a positive relationship with default.
- Loan amount showed a statistically significant but comparatively weak linear association with default.
- Home ownership showed a statistically significant categorical association with default.
- Loan grade will be evaluated as an important categorical risk segmentation variable.

### Limitations

These tests are primarily univariate or bivariate analyses.

They do not account for interactions or confounding variables.

For example, the relationship between interest rate and default may partly reflect loan grade or other risk characteristics.

Therefore, statistical analysis should not be used as the final feature-selection mechanism.

The next stage will use feature engineering and multivariable machine-learning models to evaluate the combined predictive power of the available variables.

# 13. Confidence Intervals

A confidence interval provides a range of plausible values for a population parameter based on a sample.

For this project, a 95% confidence interval will be used.

A 95% confidence interval should not be interpreted as:

> "There is a 95% probability that the true parameter is inside this particular interval."

Instead, under repeated sampling and the assumptions of the method, approximately 95% of intervals constructed using this procedure would contain the true population parameter.

Confidence intervals provide information about both the estimated value and the uncertainty surrounding that estimate.

## 13.1 Confidence Interval — Difference in Loan-to-Income Ratio

We will estimate a 95% confidence interval for the difference between the mean loan-to-income ratio of defaulted and non-defaulted applicants.

The difference is defined as:

**Default mean − Non-default mean**

In [34]:
mean_difference = (
    default.mean() - no_default.mean()
)

se_difference = np.sqrt(
    (
        default.var() / len(default)
    ) +
    (
        no_default.var() / len(no_default)
    )
)

df_welch = (
    (
        default.var() / len(default)
        +
        no_default.var() / len(no_default)
    ) ** 2
) / (
    (
        (default.var() / len(default)) ** 2
        / (len(default) - 1)
    )
    +
    (
        (no_default.var() / len(no_default)) ** 2
        / (len(no_default) - 1)
    )
)

confidence_level = 0.95
alpha_ci = 1 - confidence_level

t_critical = stats.t.ppf(
    1 - alpha_ci / 2,
    df_welch
)

margin_of_error = (
    t_critical * se_difference
)

ci_lower = (
    mean_difference - margin_of_error
)

ci_upper = (
    mean_difference + margin_of_error
)

print("Mean difference:", mean_difference)
print("95% CI lower:", ci_lower)
print("95% CI upper:", ci_upper)

Mean difference: 0.09811269742842754
95% CI lower: 0.0948547407525489
95% CI upper: 0.10137065410430618


## 13.2 Confidence Interval Interpretation

The confidence interval provides an estimated range for the population-level difference in mean loan-to-income ratio between defaulted and non-defaulted applicants.

The difference is defined as:

**Default − Non-default**

If the entire confidence interval is above zero, this provides evidence that the default group has a higher mean loan-to-income ratio.

The width of the interval provides information about estimation uncertainty.

A narrower interval indicates greater precision, while a wider interval indicates greater uncertainty.

In [35]:
interest_mean_difference = (
    interest_default.mean()
    - interest_no_default.mean()
)

interest_se = np.sqrt(
    (
        interest_default.var()
        / len(interest_default)
    )
    +
    (
        interest_no_default.var()
        / len(interest_no_default)
    )
)

interest_df = (
    (
        interest_default.var() / len(interest_default)
        +
        interest_no_default.var() / len(interest_no_default)
    ) ** 2
) / (
    (
        (interest_default.var() / len(interest_default)) ** 2
        / (len(interest_default) - 1)
    )
    +
    (
        (interest_no_default.var() / len(interest_no_default)) ** 2
        / (len(interest_no_default) - 1)
    )
)

interest_t_critical = stats.t.ppf(
    0.975,
    interest_df
)

interest_margin = (
    interest_t_critical * interest_se
)

interest_ci_lower = (
    interest_mean_difference
    - interest_margin
)

interest_ci_upper = (
    interest_mean_difference
    + interest_margin
)

print(
    "Mean difference:",
    interest_mean_difference
)

print(
    "95% CI lower:",
    interest_ci_lower
)

print(
    "95% CI upper:",
    interest_ci_upper
)

Mean difference: 2.387312430040822
95% CI lower: 2.305125410280056
95% CI upper: 2.4694994498015883


## 14. Final Statistical Analysis Summary

The statistical analysis combined hypothesis testing, correlation analysis, and effect-size evaluation.

The objective was not simply to identify statistically significant relationships, but also to evaluate their practical magnitude.

The analysis demonstrates that:

- Loan-to-income ratio shows a substantial difference between defaulted and non-defaulted applicants.
- Interest rate is positively associated with default.
- Loan amount has a statistically significant but weak linear association with default.
- Home ownership is statistically associated with default.
- Loan grade is being evaluated as a categorical risk-segmentation variable.

### Statistical Interpretation Principle

Statistical tests provide evidence about relationships in the observed data.

They do not prove causation and should not be treated as standalone feature-selection methods.

The final predictive model will evaluate multiple variables simultaneously and will be assessed using appropriate classification metrics.

# 15. Notebook Conclusion

The Statistical Analysis stage has formally evaluated several relationships identified during EDA.

### Methods Used

- Welch's independent two-sample t-test
- Chi-Square Test of Independence
- Pearson correlation significance testing
- Cohen's d
- Cramér's V
- Confidence intervals
- Multiple-testing considerations

### Key Learning

A strong statistical analysis should answer three separate questions:

1. **Does a relationship exist?**
   - Statistical significance / p-value

2. **How strong is the relationship?**
   - Effect size

3. **How precisely is the effect estimated?**
   - Confidence interval

These concepts provide a stronger foundation for machine-learning feature engineering and model development.

### Transition to Machine Learning

The next stage will transform the statistically and analytically understood data into model-ready features.

This will include:

- Encoding categorical variables
- Handling remaining missing values
- Feature transformation
- Feature selection
- Train/validation/test splitting
- Preventing data leakage
- Feature scaling
- Handling class imbalance
- Building reusable preprocessing pipelines

### Next Notebook

**05_Feature_Engineering.ipynb**